# BMI Calculation Workflow with LangGraph
A clean, modular 2-node graph to calculate BMI and determine weight category.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

In [2]:
class BMIState(TypedDict, total=False):
    weight_kg: float
    height_m: float
    bmi: float
    bmi_category: str

In [3]:
def calculate_bmi(state: BMIState) -> dict:
    """Calculates BMI from weight (kg) and height (m)."""
    weight = state["weight_kg"]
    height = state["height_m"]
    bmi = round(weight / (height ** 2), 2)
    return {"bmi": bmi}


def categorize_bmi(state: BMIState) -> dict:
    """Categorizes BMI into standard WHO ranges."""
    bmi = state["bmi"]
    if bmi < 18.5:
        category = "Underweight"
    elif 18.5 <= bmi < 25.0:
        category = "Normal"
    elif 25.0 <= bmi < 30.0:
        category = "Overweight"
    else:
        category = "Obese"
    return {"bmi_category": category}

In [4]:
# Build workflow graph
builder = StateGraph(BMIState)

# Add nodes
builder.add_node("calculate_bmi", calculate_bmi)
builder.add_node("categorize_bmi", categorize_bmi)

# Add edges
builder.add_edge(START, "calculate_bmi")
builder.add_edge("calculate_bmi", "categorize_bmi")
builder.add_edge("categorize_bmi", END)

# Compile the graph
app = builder.compile()

In [5]:
# Visualize workflow graph
display(Image(app.get_graph().draw_mermaid_png()))

In [6]:
# Invoke graph with sample inputs
initial_state = {"weight_kg": 80.0, "height_m": 1.73}
output = app.invoke(initial_state)

print(f"Weight: {output['weight_kg']} kg")
print(f"Height: {output['height_m']} m")
print(f"BMI: {output['bmi']}")
print(f"Category: {output['bmi_category']}")

Weight: 80.0 kg
Height: 1.73 m
BMI: 26.73
Category: Overweight
